# Singular Value Decomposition (SVD)

## Overview

Singular Value Decomposition is a powerful matrix factorization technique that works for **rectangular matrices**. Unlike eigenvalue decomposition which requires square matrices, SVD can decompose any m × n matrix.

## Theory

For any matrix **M** of size m × n:

$$\mathbf{M} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$$

Where:
- **U** = [u₁, u₂, ..., u_m] is m × m matrix of **left singular vectors** (orthonormal)
- **Σ** = diagonal matrix of **singular values** σ₁ ≥ σ₂ ≥ ... ≥ σ_r ≥ 0, where r = min(m, n)
- **V** = [v₁, v₂, ..., v_n] is n × n matrix of **right singular vectors** (orthonormal)

### Key Properties:
- Singular values are always non-negative and real (even if M has complex entries)
- U and V are orthogonal matrices: U^T U = I, V^T V = I
- Singular vectors form r-dimensional orthonormal basis vectors

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

print("Required libraries imported successfully")

## 1. Basic SVD Computation

In [ ]:
# Create a rectangular matrix (3x2)
M = np.array([[1, 2],
              [3, 4],
              [5, 6]], dtype=float)

print("Original Matrix M (3x2):")
print(M)
print(f"\nMatrix shape: {M.shape}")

# Compute SVD
U, Sigma, VT = np.linalg.svd(M)

print("\n" + "="*50)
print("SVD Components:")
print("="*50)

print(f"\nU shape: {U.shape}")
print("U (left singular vectors):")
print(U)

print(f"\nSingular values (Sigma): {Sigma}")
print(f"Sigma shape: {Sigma.shape}")

print(f"\nVT shape: {VT.shape}")
print("VT (right singular vectors transposed):")
print(VT)

# Verify reconstruction
print("\n" + "="*50)
print("Reconstruction:")
print("="*50)

# We need to handle the rectangular Sigma
Sigma_full = np.zeros((U.shape[1], VT.shape[0]))
np.fill_diagonal(Sigma_full, Sigma)

M_reconstructed = U @ Sigma_full @ VT
print("\nReconstructed M:")
print(M_reconstructed)

print(f"\nReconstruction error: {np.linalg.norm(M - M_reconstructed):.2e}")

## 2. Orthogonality Properties

In [ ]:
# Verify orthogonality of U
print("U^T @ U (should be identity):")
print(U.T @ U)
print(f"Is U orthogonal? {np.allclose(U.T @ U, np.eye(U.shape[1]))}")

# Verify orthogonality of VT (equivalently, V)
print("\nVT @ VT^T (should be identity):")
print(VT @ VT.T)
print(f"Is VT orthogonal? {np.allclose(VT @ VT.T, np.eye(VT.shape[0]))}")

# Check singular vectors are orthonormal
print("\n" + "="*50)
print("Singular Vectors (Orthonormal):")
print("="*50)

# Left singular vectors (columns of U)
print("\nLeft singular vectors (columns of U):")
for i in range(U.shape[1]):
    u_i = U[:, i]
    norm = np.linalg.norm(u_i)
    print(f"  u_{i+1}: norm = {norm:.4f}")

# Right singular vectors (rows of VT, or columns of V)
print("\nRight singular vectors (rows of VT):")
for i in range(VT.shape[0]):
    v_i = VT[i, :]
    norm = np.linalg.norm(v_i)
    print(f"  v_{i+1}: norm = {norm:.4f}")

## 3. Relationship with Eigenvalue Decomposition

In [ ]:
# Relationship 1: M^T M = V Sigma^2 V^T
MTM = M.T @ M

print("Eigenvalue decomposition of M^T M:")
evals_MTM, evecs_MTM = np.linalg.eig(MTM)

# Sort by eigenvalues in descending order
idx = np.argsort(evals_MTM)[::-1]
evals_MTM_sorted = evals_MTM[idx]
evecs_MTM_sorted = evecs_MTM[:, idx]

print(f"\nEigenvalues of M^T M: {evals_MTM_sorted}")
print(f"Singular values squared: {Sigma**2}")
print(f"Match? {np.allclose(evals_MTM_sorted, Sigma**2)}")

print(f"\nEigenvectors of M^T M match V? {np.allclose(np.abs(evecs_MTM_sorted), np.abs(VT.T))}")

# Relationship 2: M M^T = U Sigma^2 U^T
MMT = M @ M.T

print("\n" + "="*50)
print("Eigenvalue decomposition of M M^T:")
print("="*50)

evals_MMT, evecs_MMT = np.linalg.eig(MMT)

# Sort by eigenvalues in descending order
idx = np.argsort(evals_MMT)[::-1]
evals_MMT_sorted = evals_MMT[idx]
evecs_MMT_sorted = evecs_MMT[:, idx]

print(f"\nEigenvalues of M M^T: {evals_MMT_sorted}")
print(f"Singular values squared (padded): {np.concatenate([Sigma**2, np.zeros(3 - len(Sigma))])}")
print(f"Match? {np.allclose(evals_MMT_sorted[:len(Sigma)], Sigma**2)}")

print(f"\nEigenvectors of M M^T match U? {np.allclose(np.abs(evecs_MMT_sorted), np.abs(U))}")

## 4. Low-Rank Approximation (Image Compression)

In [ ]:
# Create a sample image-like data
np.random.seed(42)
Image = np.random.randn(100, 80)

print(f"Original image shape: {Image.shape}")
print(f"Total elements: {Image.size}")

# Compute SVD
U_img, Sigma_img, VT_img = np.linalg.svd(Image, full_matrices=False)

print(f"\nNumber of singular values: {len(Sigma_img)}")
print(f"Singular values (first 10): {Sigma_img[:10]}")

# Calculate cumulative explained variance
cumsum_sigma_sq = np.cumsum(Sigma_img**2)
total_variance = cumsum_sigma_sq[-1]
cumulative_var = cumsum_sigma_sq / total_variance

# Find k for different compression levels
print("\nCompression analysis:")
for target_var in [0.50, 0.70, 0.90, 0.95, 0.99]:
    k = np.argmax(cumulative_var >= target_var) + 1
    original_size = Image.size
    compressed_size = (U_img[:, :k].size + Sigma_img[:k].size + VT_img[:k, :].size)
    compression_ratio = original_size / compressed_size
    print(f"  {target_var*100:.0f}% variance: k={k}, compression ratio = {compression_ratio:.2f}x")

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].semilogy(Sigma_img[:50])
axes[0, 0].set_title('Singular Values (log scale)')
axes[0, 0].set_xlabel('Index')
axes[0, 0].set_ylabel('Singular Value')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(cumulative_var)
axes[0, 1].set_title('Cumulative Explained Variance')
axes[0, 1].set_xlabel('Number of Components')
axes[0, 1].set_ylabel('Cumulative Variance Explained')
axes[0, 1].axhline(y=0.9, color='r', linestyle='--', label='90% variance')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Reconstruct with different k values
k_values = [5, 10, 20, 50]
for idx, k in enumerate(k_values):
    Image_k = U_img[:, :k] @ np.diag(Sigma_img[:k]) @ VT_img[:k, :]
    error = np.linalg.norm(Image - Image_k) / np.linalg.norm(Image)
    
    ax = axes.flat[idx + 2]
    ax.imshow(Image_k, cmap='gray')
    ax.set_title(f'k={k} (error={error:.4f})')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Geometric Interpretation of SVD

In [ ]:
# Create a 2x2 matrix for geometric visualization
A = np.array([[3, 1],
              [1, 3]], dtype=float)

print(f"Matrix A:\n{A}")

# SVD decomposition
U, Sigma, VT = np.linalg.svd(A)

print(f"\nLeft singular vectors (U):\n{U}")
print(f"\nSingular values: {Sigma}")
print(f"\nRight singular vectors (V^T):\n{VT}")

# Create unit circle
theta = np.linspace(0, 2*np.pi, 100)
unit_circle = np.array([np.cos(theta), np.sin(theta)])

# Apply transformations step by step
# 1. V^T: rotation
rotated = VT @ unit_circle

# 2. V^T @ Sigma: rotation + scaling
scaled = (VT @ np.diag(Sigma)) @ unit_circle

# 3. U @ Sigma @ V^T: full transformation
transformed = A @ unit_circle

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# Original unit circle
ax = axes[0, 0]
ax.plot(unit_circle[0], unit_circle[1], 'b-', linewidth=2, label='Unit circle')
ax.arrow(0, 0, 1, 0, head_width=0.1, head_length=0.1, fc='r', ec='r')
ax.arrow(0, 0, 0, 1, head_width=0.1, head_length=0.1, fc='g', ec='g')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Original: Unit Circle')
ax.legend()

# After V^T (rotation)
ax = axes[0, 1]
ax.plot(rotated[0], rotated[1], 'b-', linewidth=2, label='After V^T (rotation)')
ax.arrow(0, 0, VT[0, 0], VT[0, 1], head_width=0.1, head_length=0.1, fc='r', ec='r')
ax.arrow(0, 0, VT[1, 0], VT[1, 1], head_width=0.1, head_length=0.1, fc='g', ec='g')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('After V^T: Rotation')
ax.legend()

# After Sigma (scaling)
ax = axes[1, 0]
ax.plot(scaled[0], scaled[1], 'b-', linewidth=2, label='After Σ (scaling)')
ax.arrow(0, 0, Sigma[0]*VT[0, 0], Sigma[0]*VT[0, 1], head_width=0.1, head_length=0.1, fc='r', ec='r')
ax.arrow(0, 0, Sigma[1]*VT[1, 0], Sigma[1]*VT[1, 1], head_width=0.1, head_length=0.1, fc='g', ec='g')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('After Σ: Scaling')
ax.legend()

# Final result: U @ Sigma @ V^T
ax = axes[1, 1]
ax.plot(transformed[0], transformed[1], 'b-', linewidth=2, label='After U (rotation)')
ax.arrow(0, 0, A[0, 0], A[1, 0], head_width=0.1, head_length=0.1, fc='r', ec='r')
ax.arrow(0, 0, A[0, 1], A[1, 1], head_width=0.1, head_length=0.1, fc='g', ec='g')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Final: A = U Σ V^T')
ax.legend()

plt.tight_layout()
plt.show()

print("\nGeometric interpretation:")
print("1. V^T rotates the unit circle")
print("2. Σ scales along the rotated axes by singular values")
print("3. U applies another rotation to the scaled result")

## 6. SVD for Solving Linear Systems

In [ ]:
# Solve: A x = b
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 10]], dtype=float)  # 3x3 matrix

b = np.array([14, 32, 51], dtype=float)

print("Problem: Solve A x = b")
print(f"A:\n{A}")
print(f"\nb: {b}")

# Method 1: Using SVD
U, Sigma, VT = np.linalg.svd(A)

# Pseudoinverse using SVD: A^+ = V Sigma^+ U^T
Sigma_inv = np.zeros((VT.shape[0], U.shape[1]))
for i in range(min(len(Sigma), VT.shape[0])):
    if Sigma[i] > 1e-10:
        Sigma_inv[i, i] = 1 / Sigma[i]

A_pinv = VT.T @ Sigma_inv @ U.T
x_svd = A_pinv @ b

print("\n" + "="*50)
print("Solution using SVD:")
print("="*50)
print(f"x = {x_svd}")
print(f"A x = {A @ x_svd}")
print(f"Error: {np.linalg.norm(b - A @ x_svd):.2e}")

# Method 2: Using numpy's solve
x_solve = np.linalg.solve(A, b)
print(f"\nSolution using np.linalg.solve: {x_solve}")
print(f"Solutions match? {np.allclose(x_svd, x_solve)}")

## 7. SVD for Overdetermined Systems (Least Squares)

In [ ]:
# Create overdetermined system (more equations than unknowns)
# A is 5x2 (5 equations, 2 unknowns)
A = np.array([[1, 1],
              [2, 1],
              [3, 1],
              [4, 1],
              [5, 1]], dtype=float)

# True solution
x_true = np.array([2, 1], dtype=float)

# Create b with some noise
np.random.seed(42)
b = A @ x_true + 0.5 * np.random.randn(5)

print("Overdetermined system: A x = b")
print(f"A shape: {A.shape}")
print(f"b: {b}")

# SVD solution to least squares
U, Sigma, VT = np.linalg.svd(A, full_matrices=False)

# x = V Sigma^+ U^T b
Sigma_inv = np.zeros((VT.shape[0], U.shape[1]))
for i in range(len(Sigma)):
    if Sigma[i] > 1e-10:
        Sigma_inv[i, i] = 1 / Sigma[i]

x_svd = VT.T @ Sigma_inv @ U.T @ b

print("\n" + "="*50)
print("Least Squares Solution (SVD):")
print("="*50)
print(f"x_svd: {x_svd}")
print(f"x_true: {x_true}")
print(f"Residual error: {np.linalg.norm(b - A @ x_svd):.4f}")

# Compare with numpy's lstsq
x_lstsq, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)
print(f"\nx_lstsq: {x_lstsq}")
print(f"Solutions match? {np.allclose(x_svd, x_lstsq)}")

# Visualize
x_range = np.linspace(0, 6, 100)

fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot of data
ax.scatter(A[:, 0], b, s=100, alpha=0.7, label='Data points (noisy)')

# Fitted line
y_fit = x_svd[0] * x_range + x_svd[1]
ax.plot(x_range, y_fit, 'r-', linewidth=2, label=f'Fitted line: y = {x_svd[0]:.2f}x + {x_svd[1]:.2f}')

# True line
y_true = x_true[0] * x_range + x_true[1]
ax.plot(x_range, y_true, 'g--', linewidth=2, label=f'True line: y = {x_true[0]:.2f}x + {x_true[1]:.2f}')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Least Squares Fitting using SVD')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 8. SVD for Underdetermined Systems

In [ ]:
# Create underdetermined system (fewer equations than unknowns)
# A is 2x5 (2 equations, 5 unknowns)
A = np.array([[1, 2, 3, 4, 5],
              [5, 4, 3, 2, 1]], dtype=float)

b = np.array([10, 10], dtype=float)

print("Underdetermined system: A x = b")
print(f"A shape: {A.shape}")
print(f"b: {b}")
print(f"\nInfinite solutions exist. SVD gives the minimum norm solution.")

# SVD solution (minimum norm)
U, Sigma, VT = np.linalg.svd(A, full_matrices=False)

# x = V Sigma^+ U^T b
Sigma_inv = np.zeros((VT.shape[0], U.shape[1]))
for i in range(len(Sigma)):
    if Sigma[i] > 1e-10:
        Sigma_inv[i, i] = 1 / Sigma[i]

x_svd = VT.T @ Sigma_inv @ U.T @ b

print("\n" + "="*50)
print("Minimum Norm Solution (SVD):")
print("="*50)
print(f"x_svd: {x_svd}")
print(f"||x_svd||: {np.linalg.norm(x_svd):.4f}")
print(f"A @ x_svd: {A @ x_svd}")
print(f"Error: {np.linalg.norm(b - A @ x_svd):.2e}")

# Another valid solution
x_alt = np.array([2, 2, 0, 0, 0], dtype=float)
print(f"\nAlternative solution: {x_alt}")
print(f"||x_alt||: {np.linalg.norm(x_alt):.4f}")
print(f"A @ x_alt: {A @ x_alt}")
print(f"Also satisfies equation? {np.allclose(A @ x_alt, b)}")

## 9. SVD for Noise Reduction

In [ ]:
# Create a clean low-rank matrix
np.random.seed(42)
r = 5  # true rank
U_true = np.random.randn(50, r)
VT_true = np.random.randn(r, 40)
M_clean = U_true @ VT_true

# Add noise
noise = 0.5 * np.random.randn(50, 40)
M_noisy = M_clean + noise

print(f"True rank: {r}")
print(f"Matrix shape: {M_clean.shape}")
print(f"SNR: {np.linalg.norm(M_clean) / np.linalg.norm(noise):.2f}")

# SVD decomposition
U, Sigma, VT = np.linalg.svd(M_noisy, full_matrices=False)

# Denoise by keeping only top-r components
M_denoised = U[:, :r] @ np.diag(Sigma[:r]) @ VT[:r, :]

print("\n" + "="*50)
print("Denoising Results:")
print("="*50)

error_noisy = np.linalg.norm(M_clean - M_noisy) / np.linalg.norm(M_clean)
error_denoised = np.linalg.norm(M_clean - M_denoised) / np.linalg.norm(M_clean)

print(f"Relative error (noisy): {error_noisy:.4f}")
print(f"Relative error (denoised): {error_denoised:.4f}")
print(f"Improvement: {error_noisy / error_denoised:.2f}x")

# Visualize singular values
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.semilogy(Sigma, 'bo-', linewidth=2, markersize=6)
ax.axvline(x=r-0.5, color='r', linestyle='--', linewidth=2, label=f'True rank = {r}')
ax.set_xlabel('Component Index')
ax.set_ylabel('Singular Value (log scale)')
ax.set_title('Singular Values of Noisy Matrix')
ax.grid(True, alpha=0.3, which='both')
ax.legend()

# Visualize first singular vectors
ax = axes[1]
ax.bar(np.arange(1, r+1), Sigma[:r], alpha=0.7, label='Kept (signal)')
ax.bar(np.arange(r+1, 20), Sigma[r:20], alpha=0.7, label='Removed (noise)', color='orange')
ax.set_xlabel('Component Index')
ax.set_ylabel('Singular Value')
ax.set_title('Singular Values: Signal vs Noise')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()

plt.tight_layout()
plt.show()

## 10. SVD Complexity and Performance

In [ ]:
import time

print("SVD Computational Complexity Analysis")
print("="*50)
print("For m x n matrix:")
print("- Full SVD: O(min(m, n)^2 * max(m, n))")
print("- Reduced SVD: O(k * min(m, n)^2 + k * max(m, n))\n")

sizes = [(100, 100), (500, 500), (1000, 1000), (100, 1000), (1000, 100)]

print("\nTiming Results:")
print("-" * 50)
print(f"{'Matrix Size':<15} {'Full SVD (ms)':<20} {'Reduced SVD (ms)'}")
print("-" * 50)

for m, n in sizes:
    M = np.random.randn(m, n)
    
    # Full SVD
    start = time.time()
    U, Sigma, VT = np.linalg.svd(M, full_matrices=True)
    time_full = (time.time() - start) * 1000
    
    # Reduced SVD
    start = time.time()
    U_red, Sigma_red, VT_red = np.linalg.svd(M, full_matrices=False)
    time_reduced = (time.time() - start) * 1000
    
    print(f"{m} x {n:<8} {time_full:<20.2f} {time_reduced:.2f}")

print("\nNote: Reduced SVD is faster when m >> n or n >> m")

## Summary

**Singular Value Decomposition (SVD)** is a fundamental linear algebra technique with applications in:

1. **Data Compression** - Low-rank approximations of images, matrices
2. **Noise Reduction** - Separating signal from noise
3. **Dimensionality Reduction** - Similar to PCA
4. **Image Processing** - Compression, reconstruction
5. **Linear Systems** - Solving overdetermined and underdetermined systems
6. **Matrix Inversion** - Computing pseudoinverse
7. **Pattern Recognition** - Identifying dominant patterns
8. **Recommender Systems** - Collaborative filtering
9. **Natural Language Processing** - Latent Semantic Analysis
10. **Machine Learning** - Feature extraction and analysis

### Key Takeaways:
- SVD works for rectangular matrices (unlike eigendecomposition)
- Singular values indicate importance of components
- Low-rank approximations provide efficient representations
- SVD provides geometric insight into matrix transformations